<a href="https://colab.research.google.com/github/hozaifbinFarid/rag-chatbot/blob/main/SmartAg_Telegram_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌾 SmartAg — Telegram Bot Integration
## IoT-WSNs + Deep Learning Smart Agriculture · Voice & Photo via Telegram
---
**Author:** Most. Azmeri Zaman Roshni &nbsp;|&nbsp; **Supervisor:** Prof. Md. Najmul Hossain  
**Dept. EECE — Pabna University of Science and Technology (PUST), Bangladesh**

---

### What this notebook does

| Telegram Input | AI Processing | Bot Output |
|---|---|---|
| 🎤 Voice message (EN / বাংলা) | Whisper STT → AgriNLP → SmartAg | 📝 Text reply + 🔊 Voice reply |
| 📸 Leaf photo | CNN MobileNetV2 (38 classes) | 🔬 Disease name + treatment + 🔊 Diagnosis |
| ✍️ Text message | AgriNLP → SmartAg pipeline | 📝 Text reply + 🔊 Voice reply |

### ▶️ Quick-start (run cells in order)
1. **Cell 2** — install packages (~2 min first run)  
2. **Cell 3** — mount Google Drive  
3. **Cells 4–10** — imports, paths, load all models  
4. **Cell 11** — paste your `@BotFather` token  
5. **Cell 12** — define handlers  
6. **Cell 13** — 🚀 launch bot  

> 💡 Keep this Colab tab open while the bot is running.  
> Click **■ Stop** on Cell 13 to shut the bot down cleanly.


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2  STEP 1 — Install required packages
# ════════════════════════════════════════════════════════════════
# Run once per Colab session.  First run takes ~2 minutes.

import subprocess, sys

PACKAGES = [
    "python-telegram-bot==20.7",   # Telegram Bot API (async v20+)
    "nest-asyncio",                # Patch Colab event loop for asyncio
    "openai-whisper",              # STT — supports Bangla + English
    "gTTS",                        # Google Text-to-Speech
    "pydub",                       # OGG → WAV audio conversion
]

for pkg in PACKAGES:
    print(f"  📦 {pkg}...")
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

# ffmpeg: required by pydub and Whisper for audio decoding
print("  🔧 ffmpeg...")
subprocess.run(["apt-get", "install", "-y", "ffmpeg"], capture_output=True)

print()
print("✅ All packages ready!")
print("   python-telegram-bot 20.7 | openai-whisper | gTTS | pydub | ffmpeg")


  📦 python-telegram-bot==20.7...
  📦 nest-asyncio...
  📦 openai-whisper...
  📦 gTTS...
  📦 pydub...
  🔧 ffmpeg...

✅ All packages ready!
   python-telegram-bot 20.7 | openai-whisper | gTTS | pydub | ffmpeg


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3  STEP 2 — Mount Google Drive
# ════════════════════════════════════════════════════════════════
# Your trained SmartAg models must be under:
#   MyDrive/SmartAg_Thesis/03_Models/

from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted at /content/drive")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4  STEP 3 — Import all libraries
# ════════════════════════════════════════════════════════════════
import os, re, json, time, asyncio, logging, warnings
import numpy as np
import nest_asyncio

# Deep learning
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array

# Machine learning
import joblib

# Image & audio
from PIL import Image
import whisper
from gtts import gTTS
from pydub import AudioSegment

# Telegram (async v20+)
from telegram import Update
from telegram.ext import (
    ApplicationBuilder, CommandHandler,
    MessageHandler, ContextTypes, filters,
)

# ── CRITICAL: patch Colab's existing event loop ──────────────────
# Without this, python-telegram-bot cannot start its async loop
# inside Colab's already-running IPython event loop.
nest_asyncio.apply()

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

logging.basicConfig(
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    level=logging.INFO
)
logger = logging.getLogger(__name__)

print(f"✅ Libraries imported")
print(f"   TensorFlow {tf.__version__} | nest_asyncio applied")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5  STEP 4 — Configure paths (matches SmartAg_Thesis layout)
# ════════════════════════════════════════════════════════════════

DRIVE_BASE  = "/content/drive/MyDrive/SmartAg_Thesis"
MODELS_DIR  = f"{DRIVE_BASE}/03_Models"

CNN_MODEL_PATH  = f"{MODELS_DIR}/plant_disease_cnn_final.h5"
CLASS_IDX_PATH  = f"{MODELS_DIR}/class_indices.json"
CROP_MODEL_PATH = f"{MODELS_DIR}/crop_rf_model.pkl"
CROP_ENC_PATH   = f"{MODELS_DIR}/crop_label_encoder.pkl"
FERT_MODEL_PATH = f"{MODELS_DIR}/fert_rf_model.pkl"
FERT_ENC_PATH   = f"{MODELS_DIR}/fert_label_encoder.pkl"

TEMP_DIR = "/tmp/smartag_bot"
os.makedirs(TEMP_DIR, exist_ok=True)

IMG_SIZE = 224
TOP_K    = 3

# ── Verify all model files exist ─────────────────────────────────
print("Checking model files on Google Drive...\n")
REQUIRED = {
    "CNN Disease Model   ": CNN_MODEL_PATH,
    "Class Indices JSON  ": CLASS_IDX_PATH,
    "Crop RF Model       ": CROP_MODEL_PATH,
    "Crop Label Encoder  ": CROP_ENC_PATH,
    "Fertilizer RF Model ": FERT_MODEL_PATH,
    "Fertilizer Encoder  ": FERT_ENC_PATH,
}
all_ok = True
for name, path in REQUIRED.items():
    exists = os.path.exists(path)
    mark   = "✅" if exists else "❌ MISSING"
    print(f"  {mark}  {name}  ({path.split("/")[-1]})")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("✅ All model files found — proceed to Cell 6!")
else:
    print("⚠️  Run your SmartAg training notebook first to generate the missing files.")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6  STEP 5 — Load all AI models into memory
# ════════════════════════════════════════════════════════════════
# CNN loads in ~30 s.  Whisper downloads ~150 MB on first run.

print("Loading models — please wait...\n")

# [1/4] CNN plant disease model
print("  [1/4] CNN  MobileNetV2 disease model...")
cnn_model    = load_model(CNN_MODEL_PATH)
with open(CLASS_IDX_PATH) as f:
    class_indices = json.load(f)
idx_to_class = {int(v): k for k, v in class_indices.items()}
NUM_CLASSES  = len(idx_to_class)
print(f"        ✅ {NUM_CLASSES} disease/health classes")

# [2/4] Crop recommendation RF
print("  [2/4] Crop recommendation (Random Forest)...")
crop_model   = joblib.load(CROP_MODEL_PATH)
crop_encoder = joblib.load(CROP_ENC_PATH)
NUM_CROPS    = len(crop_encoder.classes_)
print(f"        ✅ {NUM_CROPS} crop classes | 200 trees")

# [3/4] Fertilizer RF
print("  [3/4] Fertilizer recommendation (Random Forest)...")
fert_model   = joblib.load(FERT_MODEL_PATH)
fert_encoder = joblib.load(FERT_ENC_PATH)
NUM_FERTS    = len(fert_encoder.classes_)
print(f"        ✅ {NUM_FERTS} fertilizer types | 200 trees")

# [4/4] Whisper STT
print("  [4/4] Whisper STT  base model (~74M params)...")
whisper_model = whisper.load_model("base")
print( "        ✅ Loaded — English + Bangla multilingual")

print()
print("━" * 52)
print("🎉 All 4 models loaded and ready!")
print(f"   CNN {NUM_CLASSES} cls | Crops {NUM_CROPS} | Ferts {NUM_FERTS} | Whisper base")
print("━" * 52)


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7  MODULE A — Plant Disease Detection  (CNN MobileNetV2)
# ════════════════════════════════════════════════════════════════

def preprocess_leaf(image_path):
    img = Image.open(image_path).convert('RGB')
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    arr = img_to_array(img) / 255.0
    return np.expand_dims(arr, axis=0)


def detect_disease(image_path):
    arr     = preprocess_leaf(image_path)
    preds   = cnn_model.predict(arr, verbose=0)[0]
    top_idx = np.argsort(preds)[::-1][:TOP_K]
    results = []
    for idx in top_idx:
        raw   = idx_to_class[idx]
        parts = raw.split('___')
        plant = parts[0].replace('_', ' ') if parts else raw
        cond  = parts[1].replace('_', ' ') if len(parts) > 1 else ''
        results.append({
            'disease'   : f'{plant} — {cond}' if cond else plant,
            'raw_label' : raw,
            'plant'     : plant,
            'condition' : cond,
            'confidence': float(preds[idx]),
            'is_healthy': 'healthy' in raw.lower(),
        })
    return results


DISEASE_ADVICE = {
    'healthy'        : '✅ Your plant looks *healthy*! No disease detected. Keep monitoring regularly.',
    'late_blight'    : '⚠️ *Late Blight* — Remove infected leaves immediately. Apply copper-based fungicide. Use drip irrigation instead of overhead watering.',
    'early_blight'   : '⚠️ *Early Blight* — Remove lower infected leaves. Apply mancozeb fungicide. Mulch around base to prevent soil splash.',
    'leaf_mold'      : '⚠️ *Leaf Mold* — Improve ventilation. Reduce humidity below 85%. Apply chlorothalonil fungicide.',
    'bacterial_spot' : '⚠️ *Bacterial Spot* — Apply copper bactericide. Avoid wetting foliage. Remove severely affected plants.',
    'powdery_mildew' : '⚠️ *Powdery Mildew* — Apply sulfur or neem oil fungicide. Improve air circulation. Reduce nitrogen over-application.',
    'mosaic_virus'   : '⚠️ *Mosaic Virus* — No chemical cure. Remove infected plants. Control aphid/whitefly vectors. Disinfect tools.',
    'rust'           : '⚠️ *Rust* — Apply triazole fungicide. Remove and burn infected parts. Avoid wetting leaves.',
    'scab'           : '⚠️ *Scab* — Apply captan fungicide. Prune for air circulation. Rake fallen leaves from soil.',
    'black_rot'      : '⚠️ *Black Rot* — Remove infected tissue. Apply copper fungicide. Improve field drainage.',
    'common_rust'    : '⚠️ *Common Rust* — Apply mancozeb or propiconazole. Plant resistant varieties next season.',
    'northern_leaf_blight': '⚠️ *Northern Leaf Blight* — Apply azoxystrobin. Practice crop rotation.',
    'default'        : '⚠️ *Disease detected.* Isolate plant. Remove visible infected parts. Consult your local agricultural extension officer.',
}


def get_disease_advice(raw_label):
    label = raw_label.lower().replace('___', ' ').replace('_', ' ')
    if 'healthy' in label:
        return DISEASE_ADVICE['healthy']
    for key in DISEASE_ADVICE:
        if key != 'default' and key.replace('_', ' ') in label:
            return DISEASE_ADVICE[key]
    return DISEASE_ADVICE['default']


def format_disease_report(top3):
    top    = top3[0]
    emoji  = '✅' if top['is_healthy'] else '🔴'
    cond   = top['condition'] or 'Healthy'
    report = (
        f'🔬 *Plant Disease Analysis Report*\n'
        f'{"─" * 32}\n\n'
        f'{emoji} *Primary Diagnosis:*\n'
        f'  🌿 *{top["plant"]}* — {cond}\n'
        f'  📊 Confidence: *{top["confidence"]:.1%}*\n\n'
        f'💊 *Management Advice:*\n'
        f'{get_disease_advice(top["raw_label"])}\n\n'
        f'📋 *Other Possibilities:*\n'
    )
    for i, p in enumerate(top3[1:], 2):
        report += f'  {i}. {p["disease"]} ({p["confidence"]:.1%})\n'
    report += f'\n_Powered by MobileNetV2 CNN · {NUM_CLASSES} classes_'
    return report


print(f"✅ Module A: Disease detection ready ({NUM_CLASSES} classes)")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 8  MODULE B — Crop & Fertilizer Recommendation (RF)
# ════════════════════════════════════════════════════════════════

SOIL_MAP = {'Sandy':0,'Loamy':1,'Black':2,'Red':3,'Clayey':4,'Clay':4}


def recommend_crops(N, P, K, temperature, humidity, ph, rainfall):
    feat  = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
    proba = crop_model.predict_proba(feat)[0]
    top3  = np.argsort(proba)[::-1][:3]
    return [{'crop': crop_encoder.classes_[i].capitalize(),
             'confidence': float(proba[i])} for i in top3]


def recommend_fertilizer(N, P, K, soil_type='Loamy', crop_name=''):
    try:
        soil_enc = SOIL_MAP.get(soil_type, 1)
        crop_enc = 0
        if crop_name:
            for i, c in enumerate(crop_encoder.classes_):
                if c.lower() == crop_name.lower():
                    crop_enc = i
                    break
        feat = np.array([[soil_enc, crop_enc, N, P, K]])
        pred = fert_model.predict(feat)[0]
        return fert_encoder.inverse_transform([pred])[0]
    except Exception as e:
        logger.warning(f'Fertilizer fallback: {e}')
        if N < 30:   return 'Urea (high nitrogen needed — N is critically low)'
        if P < 20:   return 'DAP — Diammonium Phosphate (P is low)'
        if K < 20:   return 'Muriate of Potash / MOP (K is low)'
        return 'NPK 20-20-20 (balanced general purpose)'


def suggest(N, P, K, temperature, humidity, ph, rainfall, soil_type='Loamy'):
    crops = recommend_crops(N, P, K, temperature, humidity, ph, rainfall)
    fert  = recommend_fertilizer(N, P, K, soil_type, crops[0]['crop'])
    return {'crops': crops, 'fertilizer': fert}


def format_crop_report(N, P, K, temperature, humidity, ph, rainfall, soil_type='Loamy'):
    result = suggest(N, P, K, temperature, humidity, ph, rainfall, soil_type)
    def bar(c): return ('█' * int(c * 10)).ljust(10, '░')
    report = (
        f'🌾 *Crop & Fertilizer Recommendation*\n'
        f'{"─" * 32}\n\n'
        f'*📊 Soil Parameters:*\n'
        f'  N={N} | P={P} | K={K} mg/kg\n'
        f'  Temp={temperature}°C | Humidity={humidity}%\n'
        f'  pH={ph} | Rainfall={rainfall}mm | Soil: {soil_type}\n\n'
        f'*🌱 Recommended Crops:*\n'
    )
    for i, c in enumerate(result['crops'], 1):
        report += f'  {i}. *{c["crop"]}*  {bar(c["confidence"])}  {c["confidence"]:.0%}\n'
    report += (
        f'\n*🧪 Suggested Fertilizer:*\n'
        f'  📦 {result["fertilizer"]}\n\n'
        f'_Powered by Random Forest · 200 trees · sqrt features_'
    )
    return report


def parse_soil_params(text):
    kv = {}
    for m in re.finditer(r'(N|P|K|temp|temperature|humidity|pH|ph|rainfall)\s*[=:]\s*([0-9.]+)', text, re.I):
        kv[m.group(1).lower()] = float(m.group(2))
    if len(kv) >= 6:
        return {
            'N': kv.get('n', 90), 'P': kv.get('p', 42), 'K': kv.get('k', 43),
            'temperature': kv.get('temp', kv.get('temperature', 22.5)),
            'humidity': kv.get('humidity', 80),
            'ph': kv.get('ph', kv.get('pH', 6.5)),
            'rainfall': kv.get('rainfall', 200),
        }
    nums = re.findall(r'[0-9]+\.?[0-9]*', text)
    if len(nums) >= 7:
        return {'N':float(nums[0]),'P':float(nums[1]),'K':float(nums[2]),
                'temperature':float(nums[3]),'humidity':float(nums[4]),
                'ph':float(nums[5]),'rainfall':float(nums[6])}
    return None


print(f"✅ Module B: Crop & fertilizer ready ({NUM_CROPS} crops, {NUM_FERTS} fertilizers)")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 9  MODULE C — IoT Sensor Alert System (WSN Threshold Logic)
# ════════════════════════════════════════════════════════════════

THRESHOLDS = {
    'temperature': (15.0,  35.0, '°C'   ),
    'humidity'   : (40.0,  90.0, '%'    ),
    'water_level': (20.0,  80.0, '%'    ),
    'n'          : (10.0, 200.0, 'mg/kg'),
    'p'          : (10.0, 100.0, 'mg/kg'),
    'k'          : (10.0, 200.0, 'mg/kg'),
}
SENSOR_ICONS = {
    'temperature':'🌡️','humidity':'💧','water_level':'🪣',
    'n':'🌿','p':'⚗️','k':'💊'
}


def check_alerts(reading_dict):
    alerts = []
    for sensor, value in reading_dict.items():
        key = sensor.lower()
        if key not in THRESHOLDS: continue
        low, high, unit = THRESHOLDS[key]
        val = float(value)
        if val < low:
            alerts.append({'sensor':sensor,'value':val,'severity':'LOW',
                'message':f'⬇️ {sensor.capitalize()} too LOW: {val}{unit}  (min {low}{unit})'})
        elif val > high:
            alerts.append({'sensor':sensor,'value':val,'severity':'HIGH',
                'message':f'⬆️ {sensor.capitalize()} too HIGH: {val}{unit}  (max {high}{unit})'})
    return alerts


def format_iot_report(reading_dict):
    alerts = check_alerts(reading_dict)
    banner = '🔴 *ALERTS DETECTED*' if alerts else '🟢 *ALL NORMAL*'
    report = f'📡 *IoT Sensor Dashboard*\n{"─"*32}\nStatus: {banner}\n\n*Live Readings:*\n'
    for sensor, value in reading_dict.items():
        key  = sensor.lower()
        icon = SENSOR_ICONS.get(key, '📊')
        unit = THRESHOLDS[key][2] if key in THRESHOLDS else ''
        report += f'  {icon} {sensor.capitalize()}: *{value}{unit}*\n'
    if alerts:
        report += f'\n*⚠️ {len(alerts)} Alert(s):*\n'
        for a in alerts: report += f'  {a["message"]}\n'
        report += '\n💡 _Take corrective action immediately!_'
    else:
        report += '\n✅ _All readings within safe thresholds._'
    return report


def parse_sensor_text(text):
    sensors = {}
    for m in re.finditer(r'(temperature|humidity|water_?level|[NPKnpk])\s*[=:]\s*([0-9.]+)', text, re.I):
        key = m.group(1).lower().replace('water ', 'water_level')
        sensors[key] = float(m.group(2))
    return sensors


print(f"✅ Module C: IoT alerts ready ({len(THRESHOLDS)} sensors)")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 10  MODULE D — AgriNLP Intent Classifier + Voice Pipeline
# ════════════════════════════════════════════════════════════════

INTENT_KW = {
    'disease'    : ['disease','sick','infected','blight','mold','mould','rust','spot',
                    'wilt','rot','lesion','symptom','unhealthy','dying','leaf problem',
                    'রোগ','পাতা','সংক্রমণ','নষ্ট','মরে'],
    'crop'       : ['crop','grow','cultivate','recommend','suitable','which crop',
                    'what to grow','best crop','ফসল','চাষ','কি লাগাবো','কোন ফসল'],
    'fertilizer' : ['fertilizer','fertiliser','npk','nutrient','feed','manure',
                    'urea','dap','potash','সার','পুষ্টি','ইউরিয়া'],
    'irrigation' : ['water','irrigation','moisture','dry','drip','sprinkler',
                    'watering schedule','পানি','সেচ','আর্দ্রতা'],
    'weather'    : ['weather','temperature','humidity','climate','heat','cold','frost',
                    'আবহাওয়া','তাপমাত্রা','ঠান্ডা','গরম'],
    'sensor'     : ['sensor','iot','reading','monitor','alert','threshold',
                    'nitrogen level','phosphorus','potassium','water level',
                    'সেন্সর','মনিটর'],
    'greeting'   : ['hello','hi','hey','good morning','assalamu','salam',
                    'নমস্কার','হ্যালো','আস্সালামুয়ালাইকুম','সুপ্রভাত'],
    'help'       : ['help','guide','what can','how to use','features',
                    'সাহায্য','সহায়তা','কি করতে পারো'],
}


def classify_intent(text):
    t      = text.lower()
    scores = {intent: sum(1 for kw in kws if kw in t) for intent, kws in INTENT_KW.items()}
    best   = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'general'


KB = {
    'irrigation': (
        '💧 *Irrigation Guidance*\n\n'
        '• Most crops need 25–50 mm water per week\n'
        '• Water early morning (6–8 AM) to reduce evaporation\n'
        '• Finger test: insert 5 cm — if dry, irrigate now\n'
        '• Drip irrigation saves 40–60% vs flood method\n'
        '• Rice: maintain 2–5 cm standing water during growing phase\n'
        '• Ensure drainage channels to prevent waterlogging'
    ),
    'weather': (
        '🌡️ *Optimal Weather for Crops*\n\n'
        '• Most crops: 15–35°C  |  Optimal: 20–28°C\n'
        '• Relative humidity: 40–90%\n'
        '• Frost risk below 5°C — protect with mulch or row covers\n'
        '• Heat stress above 38°C reduces photosynthesis & yield\n'
        '• Bangladesh avg temp: 17°C (Jan) → 34°C (May)'
    ),
    'greeting': (
        '🌾 *Assalamu Alaikum! Welcome to SmartAg Bot* 🇧🇩\n\n'
        'আমি আপনার AI কৃষি সহকারী!\n\n'
        '*I can help with:*\n'
        '🔬 *Disease Detection* — Send a leaf photo\n'
        '🌱 *Crop Recommendation* — Based on soil & climate data\n'
        '🧪 *Fertilizer Advice* — Best NPK for your crop\n'
        '💧 *Irrigation Guide* — Smart watering schedules\n'
        '📡 *Sensor Alerts* — IoT threshold monitoring\n\n'
        'Send 🎤 voice, 📸 leaf photo, or ✍️ text — I will respond in both text and voice!'
    ),
    'help': (
        '📖 *SmartAg Bot — Usage Guide*\n\n'
        '🎤 *Voice Message (EN or বাংলা):*\n'
        '  Speak your question → Whisper transcribes → AI answers → gTTS voice reply\n\n'
        '📸 *Leaf Photo:*\n'
        '  Clear photo of affected leaf → CNN detects disease → advice + voice\n\n'
        '✍️ *Text:*\n'
        '  Type any farming question → instant AI reply + voice\n\n'
        '*Example queries:*\n'
        '  → My tomato leaves have dark brown spots\n'
        '  → N=90 P=42 K=43 temp=22 humidity=80 pH=6.5 rainfall=200\n'
        '  → Irrigation schedule for rice\n'
        '  → Sensor: temp=38 humidity=92 N=150\n\n'
        '/start  /help  /status'
    ),
    'general': (
        '🤔 *I did not quite understand that.*\n\n'
        'Try asking about:\n'
        '🌿 Plant disease — send a leaf photo\n'
        '🌾 Crop recommendation — provide soil parameters\n'
        '🧪 Fertilizer advice\n'
        '💧 Irrigation scheduling\n'
        '📡 Sensor monitoring\n\n'
        'Type /help for full usage guide.'
    ),
}


def build_response(text, intent):
    if intent in KB:
        return KB[intent]
    if intent in ('crop', 'fertilizer'):
        params = parse_soil_params(text)
        if params:
            soil = 'Loamy'
            for st in SOIL_MAP:
                if st.lower() in text.lower():
                    soil = st
                    break
            return format_crop_report(soil_type=soil, **params)
        return (
            '🌾 *Please provide soil parameters:*\n\n'
            'Format: `N=90 P=42 K=43 temp=22 humidity=80 pH=6.5 rainfall=200`\n\n'
            'Or just type 7 numbers in order:\n'
            '`N  P  K  temperature  humidity  pH  rainfall`'
        )
    if intent == 'sensor':
        readings = parse_sensor_text(text)
        if readings:
            return format_iot_report(readings)
        return (
            '📡 *Provide sensor readings:*\n\n'
            'Format: `temp=38 humidity=92 N=150 P=80 K=120`\n\n'
            'I will check against safe IoT thresholds.'
        )
    if intent == 'disease':
        return (
            '🔬 *For disease detection, send a leaf photo!*\n\n'
            '📸 Take a clear, well-lit photo of the affected leaf and send it here.\n'
            'The MobileNetV2 CNN will analyse it immediately.'
        )
    return KB['general']


def strip_md(text):
    text = re.sub(r'[*_`#~|]', '', text)
    text = re.sub(r'\n+', '. ', text)
    return re.sub(r' {2,}', ' ', text).strip()


def text_to_speech(text, lang='en', path=None):
    if path is None:
        path = os.path.join(TEMP_DIR, f'reply_{int(time.time()*1000)}.mp3')
    gTTS(text=strip_md(text), lang=lang, slow=False).save(path)
    return path


def ogg_to_wav(ogg_path):
    wav_path = ogg_path.rsplit('.', 1)[0] + '.wav'
    audio = AudioSegment.from_file(ogg_path, format='ogg')
    audio = audio.set_channels(1).set_frame_rate(16000)
    audio.export(wav_path, format='wav')
    return wav_path


def voice_pipeline(input_text=None, audio_path=None, lang='en'):
    # Step 1: Transcribe if audio given
    if audio_path:
        result     = whisper_model.transcribe(audio_path, task='transcribe')
        transcript = result['text'].strip()
        detected   = result.get('language', lang)
        tts_lang   = 'bn' if detected == 'bn' else 'en'
        logger.info(f'Whisper: "{transcript[:60]}" (lang={detected})')
    else:
        transcript = (input_text or '').strip()
        tts_lang   = lang
    # Step 2: Intent
    intent = classify_intent(transcript)
    logger.info(f'Intent: {intent}')
    # Step 3: Response
    response_text = build_response(transcript, intent)
    # Step 4: TTS
    mp3_path = os.path.join(TEMP_DIR, f'vp_{int(time.time()*1000)}.mp3')
    text_to_speech(response_text, lang=tts_lang, path=mp3_path)
    return {
        'transcript'   : transcript,
        'intent'       : intent,
        'response_text': response_text,
        'audio_path'   : mp3_path,
        'tts_lang'     : tts_lang,
    }


print("✅ Module D: NLP + voice pipeline ready")
print(f"   Intents: {list(INTENT_KW.keys()) + ['general']}")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 11  STEP 6 — Set your Telegram Bot Token   ← EDIT THIS
# ════════════════════════════════════════════════════════════════
#
#  How to get your token:
#  1. Open Telegram → search @BotFather
#  2. Send: /newbot
#  3. Choose a name + username for your bot
#  4. Copy the token BotFather gives you
#  5. Paste it below between the quotes
#
# ─────────────────────────────────────────────────────────────────

BOT_TOKEN = "YOUR_BOT_TOKEN_HERE"   # ← PASTE YOUR TOKEN HERE

# ─────────────────────────────────────────────────────────────────
if BOT_TOKEN == "YOUR_BOT_TOKEN_HERE":
    print("⚠️  Token not set — edit this cell and paste your @BotFather token")
elif ":" not in BOT_TOKEN or len(BOT_TOKEN) < 30:
    print("⚠️  Token format looks wrong. Expected: 1234567890:ABC-DEFxyz...")
else:
    bot_id = BOT_TOKEN.split(':')[0]
    print(f"✅ Token set  |  Bot ID: {bot_id}")
    print(f"   Preview: {BOT_TOKEN[:12]}...{BOT_TOKEN[-6:]}")
    print("   Ready to launch — proceed to Cell 12 then Cell 13!")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 12  STEP 7 — Define all Telegram Bot handlers
# ════════════════════════════════════════════════════════════════

# ─── /start ──────────────────────────────────────────────────────
async def cmd_start(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        '🌾 *SmartAg AI — Agricultural Assistant*\n'
        '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n'
        'Powered by: Deep Learning + IoT + Voice AI\n\n'
        '🎤 *Voice* — Speak EN or বাংলা → AI answers with voice\n'
        '📸 *Leaf Photo* — CNN detects disease → advice + voice\n'
        '✍️ *Text* — Type any farming question → instant reply\n\n'
        '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        'Commands: /help · /status\n\n'
        '_PUST SmartAg Thesis — Bangladesh_ 🇧🇩',
        parse_mode='Markdown'
    )


async def cmd_help(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(KB['help'], parse_mode='Markdown')


async def cmd_status(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        '⚙️ *SmartAg System Status*\n'
        '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n'
        f'✅ CNN Disease Model    —  MobileNetV2 · {NUM_CLASSES} classes\n'
        f'✅ Crop RF Model        —  {NUM_CROPS} crops · 200 trees\n'
        f'✅ Fertilizer RF Model  —  {NUM_FERTS} types · 200 trees\n'
        '✅ Whisper STT          —  base model · EN + BN\n'
        '✅ gTTS                 —  English + Bangla synthesis\n'
        f'✅ IoT Alert System     —  {len(THRESHOLDS)} sensors monitored\n\n'
        '🟢 *All systems operational*\n\n'
        '_Platform: Google Colab + Telegram Bot API v20_',
        parse_mode='Markdown'
    )


# ─── 🎤 VOICE HANDLER ────────────────────────────────────────────
async def handle_voice(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    msg_id   = update.message.message_id
    user     = update.effective_user.first_name or 'Farmer'
    ogg_path = os.path.join(TEMP_DIR, f'voice_{msg_id}.ogg')
    wav_path = None
    mp3_path = None
    try:
        await update.message.reply_text('🎤 Voice received! Transcribing with Whisper AI...')

        # Download OGG from Telegram servers
        vf = await update.message.voice.get_file()
        await vf.download_to_drive(ogg_path)
        kb = os.path.getsize(ogg_path) / 1024
        logger.info(f'Voice from {user}: {kb:.1f} KB')

        # Convert OGG → WAV (16kHz mono — optimal for Whisper)
        wav_path = ogg_to_wav(ogg_path)

        # Full voice pipeline: Whisper → NLP → SmartAg → gTTS
        t0     = time.time()
        result = voice_pipeline(audio_path=wav_path)
        elapsed = time.time() - t0
        lang_label = 'বাংলা' if result['tts_lang'] == 'bn' else 'English'

        # Send what was heard
        transcript_text = result['transcript'] or '(could not transcribe)'
        await update.message.reply_text(
            f'🗣️ *You said ({lang_label}):*\n_{transcript_text}_',
            parse_mode='Markdown'
        )

        # Send text response
        await update.message.reply_text(result['response_text'], parse_mode='Markdown')

        # Send voice response
        mp3_path = result['audio_path']
        with open(mp3_path, 'rb') as f:
            await update.message.reply_voice(
                voice=f,
                caption=f'🔊 Voice reply ({lang_label}) · {elapsed:.1f}s'
            )
        logger.info(f'Voice reply sent (msg_id={msg_id}, {elapsed:.2f}s)')

    except Exception as e:
        logger.error(f'handle_voice error: {e}', exc_info=True)
        await update.message.reply_text(
            f'❌ *Error processing voice*\n`{type(e).__name__}: {str(e)[:250]}`\n\n'
            '💡 _Try typing your question, or resend the voice message._',
            parse_mode='Markdown'
        )
    finally:
        for p in [ogg_path, wav_path, mp3_path]:
            try:
                if p and os.path.exists(p): os.remove(p)
            except Exception: pass


# ─── 📸 PHOTO HANDLER (leaf disease detection) ───────────────────
async def handle_photo(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    msg_id   = update.message.message_id
    user     = update.effective_user.first_name or 'Farmer'
    img_path = os.path.join(TEMP_DIR, f'leaf_{msg_id}.jpg')
    mp3_path = os.path.join(TEMP_DIR, f'diag_{msg_id}.mp3')
    try:
        await update.message.reply_text('📸 Leaf photo received!\n🔬 Running CNN disease analysis...')

        # Download highest-resolution version (photo[-1] is always biggest)
        pf = await update.message.photo[-1].get_file()
        await pf.download_to_drive(img_path)
        kb = os.path.getsize(img_path) / 1024
        logger.info(f'Photo from {user}: {kb:.1f} KB')

        # CNN inference
        t0      = time.time()
        top3    = detect_disease(img_path)
        elapsed = time.time() - t0
        top     = top3[0]
        logger.info(f'Disease: {top["disease"]} ({top["confidence"]:.1%}) in {elapsed:.2f}s')

        # Send text disease report
        report = format_disease_report(top3)
        await update.message.reply_text(
            report + f'\n\n⏱ _CNN inference: {elapsed:.2f}s_',
            parse_mode='Markdown'
        )

        # Build clean TTS text (no markdown)
        if top['is_healthy']:
            disease_spoken = 'No disease found — the plant is healthy.'
        else:
            disease_spoken = f'{top["condition"]} detected in {top["plant"]}.'
        tts_text = (
            f'Disease analysis complete. {disease_spoken} '
            f'Confidence: {int(top["confidence"]*100)} percent. '
            f'{strip_md(get_disease_advice(top["raw_label"]))}'
        )

        # Send voice diagnosis
        mp3_path = text_to_speech(tts_text, lang='en', path=mp3_path)
        with open(mp3_path, 'rb') as f:
            await update.message.reply_voice(
                voice=f,
                caption=f'🔊 Voice diagnosis · CNN {top["confidence"]:.0%} confidence'
            )
        logger.info(f'Photo reply sent (msg_id={msg_id})')

    except Exception as e:
        logger.error(f'handle_photo error: {e}', exc_info=True)
        await update.message.reply_text(
            f'❌ *Error analysing image*\n`{type(e).__name__}: {str(e)[:250]}`\n\n'
            '*Tips:* Clear leaf photo · Good lighting · Leaf fills frame · JPG or PNG',
            parse_mode='Markdown'
        )
    finally:
        for p in [img_path, mp3_path]:
            try:
                if p and os.path.exists(p): os.remove(p)
            except Exception: pass


# ─── ✍️ TEXT HANDLER ─────────────────────────────────────────────
async def handle_text(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    msg_id   = update.message.message_id
    mp3_path = os.path.join(TEMP_DIR, f'txt_{msg_id}.mp3')
    try:
        user_text = update.message.text.strip()
        logger.info(f'Text: {user_text[:80]}')
        await ctx.bot.send_chat_action(chat_id=update.effective_chat.id, action='typing')

        result = voice_pipeline(input_text=user_text)
        await update.message.reply_text(result['response_text'], parse_mode='Markdown')

        mp3_path = result['audio_path']
        lang_label = 'বাংলা' if result['tts_lang'] == 'bn' else 'English'
        with open(mp3_path, 'rb') as f:
            await update.message.reply_voice(voice=f, caption=f'🔊 Voice reply ({lang_label})')

    except Exception as e:
        logger.error(f'handle_text error: {e}', exc_info=True)
        await update.message.reply_text(
            f'❌ Error: `{str(e)[:250]}`\n_Please rephrase and try again._',
            parse_mode='Markdown'
        )
    finally:
        try:
            if mp3_path and os.path.exists(mp3_path): os.remove(mp3_path)
        except Exception: pass


# ─── 📎 DOCUMENT HANDLER (image sent as file) ────────────────────
async def handle_document(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    msg_id   = update.message.message_id
    img_path = os.path.join(TEMP_DIR, f'doc_{msg_id}.jpg')
    mp3_path = os.path.join(TEMP_DIR, f'docr_{msg_id}.mp3')
    try:
        doc = update.message.document
        if doc.mime_type and doc.mime_type.startswith('image/'):
            await update.message.reply_text('📎 Image file received — analysing leaf...')
            f2 = await doc.get_file()
            await f2.download_to_drive(img_path)
            top3   = detect_disease(img_path)
            report = format_disease_report(top3)
            await update.message.reply_text(report, parse_mode='Markdown')
            top  = top3[0]
            if top['is_healthy']:
                disease_spoken = 'Plant is healthy.'
            else:
                disease_spoken = f'{top["condition"]} detected in {top["plant"]}.'
            mp3_path = text_to_speech(
                f'Analysis complete. {disease_spoken} '
                f'Confidence {int(top["confidence"]*100)} percent.',
                lang='en', path=mp3_path
            )
            with open(mp3_path, 'rb') as f:
                await update.message.reply_voice(voice=f, caption='🔊 Diagnosis')
        else:
            await update.message.reply_text(
                '📎 Please send a *leaf image* (JPG or PNG).\n'
                'You can send it as a photo or as a file.',
                parse_mode='Markdown'
            )
    except Exception as e:
        logger.error(f'handle_document error: {e}', exc_info=True)
        await update.message.reply_text(f'❌ Error: `{str(e)[:250]}`', parse_mode='Markdown')
    finally:
        for p in [img_path, mp3_path]:
            try:
                if p and os.path.exists(p): os.remove(p)
            except Exception: pass


# ─── GLOBAL ERROR HANDLER ────────────────────────────────────────
async def error_handler(update: object, ctx: ContextTypes.DEFAULT_TYPE):
    logger.error(f'Unhandled error: {ctx.error}', exc_info=ctx.error)
    if isinstance(update, Update) and update.effective_message:
        try:
            await update.effective_message.reply_text(
                '⚠️ An unexpected error occurred. Please try again.'
            )
        except Exception: pass


print("✅ All Telegram handlers defined!")
print("   Commands : /start | /help | /status")
print("   Messages : 🎤 voice | 📸 photo | ✍️ text | 📎 document")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 13  STEP 8 — Launch the SmartAg Telegram Bot 🚀
# ════════════════════════════════════════════════════════════════
#
#  ⚠️  Set BOT_TOKEN in Cell 11 BEFORE running this!
#
#  To STOP the bot:
#     Click  ■ (Stop)  next to this cell
#     OR:  Runtime → Interrupt Execution  (Ctrl + M + I)
#
# ════════════════════════════════════════════════════════════════

async def start_smartag_bot():
    if BOT_TOKEN == "YOUR_BOT_TOKEN_HERE":
        print("❌ ERROR: Bot token not set!")
        print("   Go to Cell 11 and paste your token from @BotFather")
        return

    print("🤖 Building SmartAg Telegram Bot...")

    app = ApplicationBuilder().token(BOT_TOKEN).build()

    # Register command handlers
    app.add_handler(CommandHandler("start",  cmd_start))
    app.add_handler(CommandHandler("help",   cmd_help))
    app.add_handler(CommandHandler("status", cmd_status))

    # Register message handlers (order = priority — specific first)
    app.add_handler(MessageHandler(filters.VOICE,                   handle_voice))
    app.add_handler(MessageHandler(filters.PHOTO,                   handle_photo))
    app.add_handler(MessageHandler(filters.Document.IMAGE,          handle_document))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_text))

    # Global error handler
    app.add_error_handler(error_handler)

    print("✅ Handlers registered!")
    print()
    print("━" * 52)
    print("  🚀  SmartAg Bot is LIVE on Telegram!")
    print("━" * 52)
    print("  📱 Open Telegram → search your bot username")
    print("  💬 Send /start to begin")
    print("  🎤 Send a voice message in English or বাংলা")
    print("  🍃 Send a leaf photo for disease detection")
    print("  ✍️  Type any farming question")
    print()
    print("  ⬛ Click STOP ■ on this cell to shut down cleanly")
    print("━" * 52)

    # Start bot with async context manager (compatible with nest_asyncio in Colab)
    async with app:
        await app.start()
        await app.updater.start_polling(
            allowed_updates     = Update.ALL_TYPES,
            drop_pending_updates = True,
        )
        try:
            await asyncio.Event().wait()   # block until cell is stopped
        except (KeyboardInterrupt, asyncio.CancelledError):
            print("\n⏹️  Stopping bot...")
        finally:
            await app.updater.stop()
            await app.stop()
            print("✅ Bot stopped cleanly. All connections closed.")


asyncio.run(start_smartag_bot())


---
## 🧪 Optional: Test Modules Locally (Before Starting the Bot)

Run the two cells below to verify pipeline outputs **without** needing Telegram.  
Useful for debugging, checking model outputs, or benchmarking inference speed.


In [ ]:
# ════════════════════════════════════════════════════════════════
# OPTIONAL TEST A — Test the voice/text pipeline with sample queries
# ════════════════════════════════════════════════════════════════

TEST_QUERIES = [
    # (label, query)
    ('Crop recommendation',  'N=90 P=42 K=43 temp=22 humidity=80 pH=6.5 rainfall=200'),
    ('Disease intent',        'My tomato leaves have dark brown spots and are wilting'),
    ('IoT sensor alert',      'Sensor: temp=38 humidity=92 N=150 P=80 K=120'),
    ('Irrigation query',      'What is the best irrigation schedule for rice paddy?'),
    ('Bangla greeting',       'আস্সালামুয়ালাইকুম, আমাকে সাহায্য করুন'),
    ('Fertilizer query',      'What fertilizer for wheat on loamy soil N=40 P=20 K=30?'),
]

print("🧪 Testing voice pipeline with sample queries\n")
print("=" * 62)

for label, query in TEST_QUERIES:
    print(f"\n[{label}]")
    print(f"  Query: {query[:70]}")
    print("─" * 62)
    result = voice_pipeline(input_text=query)
    print(f"  Intent   : {result['intent']}")
    print(f"  TTS lang : {result['tts_lang']}")
    preview = result['response_text'].replace("*","").replace("_","")[:200]
    print(f"  Response : {preview}...")
    if result['audio_path'] and os.path.exists(result['audio_path']):
        os.remove(result['audio_path'])

print()
print("=" * 62)
print("✅ Pipeline test complete!")
print("   If results look correct → proceed to Cell 11 and start the bot.")


In [ ]:
# ════════════════════════════════════════════════════════════════
# OPTIONAL TEST B — Test CNN disease detection on a local image
# ════════════════════════════════════════════════════════════════
# Set TEST_IMAGE_PATH to any leaf image you have on Drive or Colab

TEST_IMAGE_PATH = (
    f"{DRIVE_BASE}/01_Datasets/plant_disease/valid/"
    "Tomato___Late_blight/image.JPG"   # ← adjust to a real file path
)

print(f"🔬 CNN Disease Detection — Local Test")
print(f"   Image: {TEST_IMAGE_PATH}\n")

if os.path.exists(TEST_IMAGE_PATH):
    t0      = time.time()
    top3    = detect_disease(TEST_IMAGE_PATH)
    elapsed = time.time() - t0

    print(f"⏱  Inference time: {elapsed:.3f}s  ← latency per image in the bot\n")
    print("Top-3 Predictions:")
    print("─" * 55)
    for i, pred in enumerate(top3, 1):
        bar = '█' * int(pred['confidence'] * 20)
        print(f"  {i}. {pred['disease']:<40}  {pred['confidence']:.1%}  {bar}")

    print("\nManagement Advice:")
    print("─" * 55)
    advice = get_disease_advice(top3[0]['raw_label'])
    print(advice.replace("*","").replace("_",""))

else:
    print(f"⚠️  Image not found: {TEST_IMAGE_PATH}")
    print()
    print("Options:")
    print("  1. Adjust TEST_IMAGE_PATH to a valid leaf image on your Drive")
    print("  2. Upload an image directly to Colab:")
    print("        from google.colab import files")
    print("        uploaded = files.upload()")
    print("        TEST_IMAGE_PATH = list(uploaded.keys())[0]")
